# 05 — EWMA Volatility Model

Module 05 estimates the volatility input required for option pricing.

For each equity leg:

\[
\sigma_t^2
=
\lambda\sigma_{t-1}^2
+
(1-\lambda)r_{t-1}^2,
\]

with:

\[
\lambda = 0.94.
\]

Annualized volatility is:

\[
\sigma_t^{ann}
=
\sqrt{252\sigma_t^2}.
\]

Under the standard EWMA forecasting assumption, expected future variance
is flat across the Module 04 convergence horizon. Therefore Module 05
supplies one current annualized volatility estimate for each stock, while
Module 04 continues to determine the statistical convergence horizon.

In [16]:
%load_ext autoreload
%autoreload 2


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



from src.volatility_model import (
    log_returns,
    ewma_variance_series,
    fit_ewma,
    forecast_pair_leg_volatilities,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Settings

In [17]:
EWMA_LAMBDA = 0.94
MIN_OBS = 60

## 2. Load equity prices and Module 04 signals

Adjust `PRICE_FILE` and `SIGNAL_FILE` if your project uses different filenames.

In [18]:
PRICE_FILE = ("data/processed/train_prices.parquet")

SIGNAL_FILE = ("data/processed/fou_current_signals.parquet")

prices = pd.read_parquet(PRICE_FILE)
current_signals = pd.read_parquet(SIGNAL_FILE)

prices.index = pd.to_datetime(prices.index)
prices = prices.sort_index()

print("Price matrix:", prices.shape)
print("Module 04 rows:", len(current_signals))
print(prices.index[:5])

Price matrix: (1759, 466)
Module 04 rows: 40
DatetimeIndex(['2016-01-04', '2016-01-05', '2016-01-06', '2016-01-07',
               '2016-01-08'],
              dtype='datetime64[ns]', name='Date', freq=None)


## 3. Keep only actionable Module 04 candidates

In [ ]:
candidates = (
    current_signals[
        current_signals["statistical_signal"] == True
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    f"Actionable statistical candidates: {len(candidates)}"
)

candidates[
    [
        "pair",
        "current_z",
        "selected_dte_trading_days",
    ]
]

Actionable statistical candidates: 9


,pair,current_z,selected_dte_trading_days
0,SHW-HD,-2.280265,61
1,MAS-LEN,-2.414675,93
2,SHW-DHI,-1.951851,84
3,WMT-SPGI,1.711591,83
4,RSG-AJG,-1.696617,86
5,MCO-MSCI,-1.941628,97
6,LYV-AXP,-2.174888,96
7,EMR-TEL,3.241486,122
8,PNR-NWSA,-1.599998,95


## 4. Inspect EWMA volatility for one equity

In [ ]:
row = candidates.iloc[0]

dep = row["dependent"]
indep = row["independent"]
horizon = int(
    row["selected_dte_trading_days"]
)

dep_returns = log_returns(
    prices[dep]
)

dep_variance = ewma_variance_series(
    returns=dep_returns,
    lambda_=EWMA_LAMBDA,
)

dep_annualized_vol = np.sqrt(
    252 * dep_variance
)

plt.figure(figsize=(10, 5))
plt.plot(
    dep_annualized_vol.index,
    dep_annualized_vol,
)
plt.xlabel("Date")
plt.ylabel("Annualized volatility")
plt.title(
    f"{dep} — EWMA volatility"
)
plt.show()

dep_result = fit_ewma(
    returns=dep_returns,
    lambda_=EWMA_LAMBDA,
    min_obs=MIN_OBS,
)

dep_result

## 5. Estimate EWMA volatility for one pair

In [ ]:
pair_result = forecast_pair_leg_volatilities(
    prices=prices,
    dependent=dep,
    independent=indep,
    horizon_days=horizon,
    lambda_=EWMA_LAMBDA,
    min_obs=MIN_OBS,
)

pair_result

## 6. Estimate volatility for every current statistical candidate

In [7]:
rows = []

for _, row in candidates.iterrows():

    dep = row["dependent"]
    indep = row["independent"]
    pair = row["pair"]

    horizon = int(
        row["selected_dte_trading_days"]
    )

    try:
        vol = forecast_pair_leg_volatilities(
            prices=prices,
            dependent=dep,
            independent=indep,
            horizon_days=horizon,
            lambda_=EWMA_LAMBDA,
            min_obs=MIN_OBS,
        )

        rows.append(
            {
                "pair": pair,
                "dependent": dep,
                "independent": indep,
                "current_z": row["current_z"],
                "direction": row["direction"],
                "convergence_horizon_trading_days":
                    horizon,
                "convergence_horizon_calendar_days":
                    row["selected_dte_calendar_days"],
                **vol,
            }
        )

    except Exception as exc:
        print(
            f"Skipped {pair}: {exc}"
        )

volatility_forecasts = pd.DataFrame(
    rows
)

volatility_forecasts

,pair,dependent,independent,current_z,direction,convergence_horizon_trading_days,convergence_horizon_calendar_days,horizon_days,lambda,dependent_annualized_volatility,independent_annualized_volatility,dependent_daily_variance,independent_daily_variance,dependent_n_obs,independent_n_obs
0,SHW-HD,SHW,HD,-2.280265,-1,61,89,61,0.94,0.290474,0.273200,0.000335,0.000296,1758,1758
1,MAS-LEN,MAS,LEN,-2.414675,-1,93,135,93,0.94,0.347549,0.338222,0.000479,0.000454,1758,1758
2,SHW-DHI,SHW,DHI,-1.951851,-1,84,122,84,0.94,0.290474,0.331990,0.000335,0.000437,1758,1758
3,WMT-SPGI,WMT,SPGI,1.711591,1,83,121,83,0.94,0.200908,0.296046,0.000160,0.000348,1758,1758
4,RSG-AJG,RSG,AJG,-1.696617,-1,86,125,86,0.94,0.185075,0.213293,0.000136,0.000181,1758,1758
5,MCO-MSCI,MCO,MSCI,-1.941628,-1,97,141,97,0.94,0.371192,0.382830,0.000547,0.000582,1758,1758
6,LYV-AXP,LYV,AXP,-2.174888,-1,96,140,96,0.94,0.357970,0.261306,0.000509,0.000271,1758,1758
7,EMR-TEL,EMR,TEL,3.241486,1,122,177,122,0.94,0.225365,0.292617,0.000202,0.000340,1758,1758
8,PNR-NWSA,PNR,NWSA,-1.599998,-1,95,138,95,0.94,0.323682,0.326893,0.000416,0.000424,1758,1758


## 7. Sanity checks

EWMA volatility must be positive and finite. The same fixed
\(\lambda=0.94\) is used for every stock.

In [8]:
display_cols = [
    "pair",
    "convergence_horizon_trading_days",
    "dependent_annualized_volatility",
    "independent_annualized_volatility",
    "lambda",
]

volatility_forecasts[
    display_cols
]

,pair,convergence_horizon_trading_days,dependent_annualized_volatility,independent_annualized_volatility,lambda
0,SHW-HD,61,0.290474,0.273200,0.94
1,MAS-LEN,93,0.347549,0.338222,0.94
2,SHW-DHI,84,0.290474,0.331990,0.94
3,WMT-SPGI,83,0.200908,0.296046,0.94
4,RSG-AJG,86,0.185075,0.213293,0.94
5,MCO-MSCI,97,0.371192,0.382830,0.94
6,LYV-AXP,96,0.357970,0.261306,0.94
7,EMR-TEL,122,0.225365,0.292617,0.94
8,PNR-NWSA,95,0.323682,0.326893,0.94


In [9]:
volatility_forecasts[
    [
        "dependent_annualized_volatility",
        "independent_annualized_volatility",
    ]
].describe()

,dependent_annualized_volatility,independent_annualized_volatility
count,9.000000,9.000000
mean,0.288077,0.301822
std,0.069597,0.049802
min,0.185075,0.213293
25%,0.225365,0.273200
50%,0.290474,0.296046
75%,0.347549,0.331990
max,0.371192,0.382830


In [10]:
assert (
    volatility_forecasts[
        "dependent_annualized_volatility"
    ] > 0
).all()

assert (
    volatility_forecasts[
        "independent_annualized_volatility"
    ] > 0
).all()

assert np.isfinite(
    volatility_forecasts[
        "dependent_annualized_volatility"
    ]
).all()

assert np.isfinite(
    volatility_forecasts[
        "independent_annualized_volatility"
    ]
).all()

assert (
    volatility_forecasts["lambda"]
    == EWMA_LAMBDA
).all()

## 8. Save Module 05 output

In [11]:
OUTPUT = ("data/processed/volatility_forecasts.parquet"
)

volatility_forecasts.to_parquet(
    OUTPUT,
    index=False,
)

print(
    f"Saved: {OUTPUT}"
)

Saved: data/processed/volatility_forecasts.parquet
